In [ ]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# Prep

Load packages and open the SFTDS database connection. The hedge-fund and ISIN universes are sourced from `Data/sftds_hedgefunds.csv` and `Data/EA_ISINs.xlsx` (or queried live from the DB).

# 1. Build the main panel

Construct the daily bond × hedge-fund-positioning panel and write `Data/monetary_policy_induced_position.csv` — the input consumed by every analysis `.do` file.

In [ ]:
df = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\bond_timeseries_v2.csv')

In [ ]:
df['Dates'] = pd.to_datetime(df['Dates'])
df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])

In [ ]:
# calculate residual maturity in years
df['residual_bond_maturity'] = ((df['bond_maturity'] - df['Dates']).dt.days / 365)

In [ ]:
df = df[df['residual_bond_maturity'] >= 0]

In [ ]:
df['amt_issued'] = (
    df['amt_issued']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace(' ', '', regex=False)
)

# convert to float, coercing invalid entries to NaN
df['amt_issued'] = pd.to_numeric(df['amt_issued'], errors='coerce')
df['amt_issued'] = df['amt_issued']/1e9

In [ ]:
df['amt_issued'].fillna(df['amt_issued'].median(), inplace=True)

In [ ]:
df['collateral_country'] = df['ISIN'].str[:2]

In [ ]:
df = df[df['collateral_country'].isin(['DE','IT'])]

In [ ]:
securities = tuple(df['ISIN'].unique())

In [ ]:
# Data prep
query = f""" 

SELECT DISTINCT lender_id

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
""" 

df_hf = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f""" 

SELECT DISTINCT borrower_id

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
""" 

df_hf_b = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f""" 

SELECT s.business_date, 
security_isin as isin,
sum(nominal_euro) as borrowing_volume,
avg(contractual_maturity) as long_maturity,
avg(haircut) as long_haircut

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
--AND contractual_maturity > 1
GROUP BY business_date, isin
ORDER BY business_date, isin 

""" 

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f""" 
 

SELECT s.business_date,
security_isin as isin,
sum(nominal_euro) as lending_volume,
avg(contractual_maturity) as short_maturity,
avg(haircut) as short_haircut

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'  
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
--AND contractual_maturity > 1
GROUP BY business_date, isin
ORDER BY business_date, isin 

""" 

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df_repo = df_borrowing.merge( df_lending, on=['business_date', 'isin'], how='outer', suffixes=('_borrowing', '_lending') )
 

In [ ]:
df_repo['business_date'] = pd.to_datetime(df_repo['business_date'])

In [ ]:
df = (
    df[['Dates', 'ISIN', 'PX_ASK', 'PX_BID', 'YLD_YTM_ASK', 'YLD_YTM_BID', 'amt_issued', 'residual_bond_maturity', 'collateral_country']]
    .merge(df_repo, right_on=['business_date','isin'], left_on = ['Dates', 'ISIN'], how='left')
)

In [ ]:
df = df[df['PX_ASK'].isna() == False]

In [ ]:
df.drop(columns=['business_date', 'isin'], inplace=True)

In [ ]:
df = df.rename(columns={'Dates': 'business_date'})
df = df.rename(columns={'ISIN': 'isin'})

In [ ]:
df['borrowing_volume'].fillna(0, inplace = True)
df['lending_volume'].fillna(0, inplace = True)

In [ ]:
df['net_pos'] = (df['borrowing_volume'] - df['lending_volume'])/1e9

In [ ]:
shocks_all = pd.read_excel('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\altavilla.xlsx', sheet_name = 'Monetary Event Window')

In [ ]:
shocks_all['date'] = pd.to_datetime(shocks_all['date'], dayfirst=True)

In [ ]:
out = (
    df.merge(shocks_all[['date', 'OIS_2Y']], left_on=['business_date'], right_on=['date'], how='left')
       .drop(columns=['date'])  # helpers
)


In [ ]:
out['OIS_2Y'].fillna(0, inplace=True)

In [ ]:
out = out.sort_values(['isin','business_date'])

# mid yield
out['yld_mid'] = (out['YLD_YTM_BID'] + out['YLD_YTM_ASK']) / 2

# ISIN-level daily change
out['delta_y'] = out.groupby('isin')['yld_mid'].diff()

In [ ]:
# 1. Base Intensity: Rolling mean of |net_pos| (Magnitude only)
out['abs_net'] = (out['net_pos'].abs() / out['amt_issued']) * 100

In [ ]:
hf_roll = (
    out.groupby('isin')['abs_net']
     .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
     .reset_index(level=0, drop=True)   # <-- align index with d
)

out['hf_intensity_pre'] = hf_roll

In [ ]:
out['net_pos_scaled'] = (out['net_pos'] / out['amt_issued']) * 100

hf_roll_signed = (
    out.groupby('isin')['net_pos_scaled']
     .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
     .reset_index(level=0, drop=True)
)

out['hf_intensity_long'] = np.where(hf_roll_signed > 0, out['hf_intensity_pre'], 0)
out['hf_intensity_short'] = np.where(hf_roll_signed < 0, out['hf_intensity_pre'], 0)

In [ ]:
out['hf_intensity_pre']= out['hf_intensity_pre'].fillna(0)

In [ ]:
out['bid_ask_spread'] = out['PX_ASK'] - out['PX_BID'] 

In [ ]:
out['delta_y'] = out['delta_y']*100

In [ ]:
ctd = pd.read_excel('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\CTD.xlsx')

In [ ]:
# month-end of (Year, Month)
ctd['period_end'] = pd.to_datetime(dict(year=ctd['Year'], month=ctd['Month'], day=1)) + pd.offsets.MonthEnd(0)
ctd['period_start'] = ctd['period_end'] - pd.DateOffset(years=1)

In [ ]:
# --- expand & mark in-window ---
m = out.merge(ctd[['isin','period_start','period_end']], on='isin', how='left')
m['in_window'] = (m['business_date'] >= m['period_start']) & (m['business_date'] <= m['period_end'])

# collapse back to one row per (ISIN, business_date): 1 if any window matches
flag_df = (m.groupby(['isin','business_date'], as_index=False)['in_window']
             .any()
             .rename(columns={'in_window':'ctd_flag'}))

# --- attach flag to original `out` ---
out = out.merge(flag_df, on=['isin','business_date'], how='left')
out['ctd_flag'] = out['ctd_flag'].fillna(False).astype(int)


In [ ]:
# create binary
out.loc[(out['hf_intensity_pre'] > 0), 'hf_involved'] = 1
out['hf_involved'].fillna(0, inplace = True)

In [ ]:
out['prev_net_pos'] = out.groupby('isin')['net_pos'].shift(1)

In [ ]:
# create binary
out.loc[(out['prev_net_pos'] > 0), 'hf_involved_long'] = 1
out['hf_involved_long'].fillna(0, inplace = True)

out.loc[(out['prev_net_pos'] < 0), 'hf_involved_short'] = 1
out['hf_involved_short'].fillna(0, inplace = True)

In [ ]:
out = out[out['hf_intensity_pre'] < 18]

In [ ]:
# Four categories: None / Low / Medium / High
out['hf_category'] = 'ANone'

# Identify ISIN–days with positive HF activity
mask = out['hf_intensity_pre'].fillna(0) > 0

# Within each country, split only those into terciles
out.loc[mask, 'hf_category'] = (
    out.loc[mask]
       .groupby('collateral_country')['hf_intensity_pre']
       .transform(lambda x: pd.qcut(x, 2, labels=['Low', 'High']))
       .astype(str)
)

In [ ]:
# Four categories: None / Low / Medium / High
out['hf_category_ext'] = 'ANone'

# Identify ISIN–days with positive HF activity
mask = out['hf_intensity_pre'].fillna(0) > 0

# Within each country, split only those into terciles
out.loc[mask, 'hf_category_ext'] = (
    out.loc[mask]
       .groupby('collateral_country')['hf_intensity_pre']
       .transform(lambda x: pd.qcut(x, 3, labels=['Low', 'Medium', 'High']))
       .astype(str)
)

In [ ]:
out = out[out['delta_y'].isna() == False]

In [ ]:
out = out[(out['delta_y'] <= 40) & (out['delta_y'] >= -40)]

In [ ]:
germany = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_DE_prices.csv')

In [ ]:
italy = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_IT_prices.csv')

In [ ]:
spain = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_ES_prices.csv')

In [ ]:
france = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_FR_prices.csv')

In [ ]:
df_duration = pd.concat([germany, italy], ignore_index=True)

In [ ]:
df_duration['refdate'] = pd.to_datetime(df_duration['refdate'])

In [ ]:
out = out.merge(df_duration[['refdate', 'bondcode', 'duration']], left_on = ['business_date', 'isin'], right_on = ['refdate', 'bondcode'], how = 'left')

In [ ]:
out['duration'].fillna(out['residual_bond_maturity'], inplace = True)

In [ ]:
jk_shock = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\shocks_ecb_mpd_me_d.csv')

In [ ]:
jk_shock['business_date'] = pd.to_datetime(jk_shock['date'])

In [ ]:
out = out.merge(jk_shock, on = 'business_date', how = 'left')

In [ ]:
# Option 1: reassign
out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']] = out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']].fillna(0)

In [ ]:
out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']] = out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']] * 100

In [ ]:
# 1. Calculate the daily change in position (Flow)
out['daily_net_change'] = out.groupby('isin')['net_pos'].diff()
out['delta_intensity'] = (out['daily_net_change'] / out['amt_issued']) * 100

In [ ]:
# 3. Determine the Direction ENTERING the shock (State) - Keep this as is
out['is_long_pre'] = (out.groupby('isin')['net_pos'].shift(1) > 0).astype(int)
out['is_short_pre'] = (out.groupby('isin')['net_pos'].shift(1) < 0).astype(int)

In [ ]:
out['placebo_shock'] = out.groupby('isin')['OIS_2Y'].shift(15)
out['placebo_shock'].fillna(0, inplace=True)

In [ ]:
# Data prep: fund-level LONG positions (HF is the borrower -> long the bond)
query = f""" 

SELECT s.business_date, 
security_isin as isin,
borrower_id as fund_id,
sum(nominal_euro) as long_vol

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, isin, borrower_id
ORDER BY business_date, isin 

""" 

df_long_fund = pd.read_sql_query(query, cnxn)
df_long_fund['business_date'] = pd.to_datetime(df_long_fund['business_date'])

In [ ]:
# Data prep: fund-level SHORT positions (HF is the lender -> short the bond)
query = f""" 

SELECT s.business_date,
security_isin as isin,
lender_id as fund_id,
sum(nominal_euro) as short_vol

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'  
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, isin, lender_id
ORDER BY business_date, isin 

""" 

df_short_fund = pd.read_sql_query(query, cnxn)
df_short_fund['business_date'] = pd.to_datetime(df_short_fund['business_date'])

In [ ]:
# --- Fund-level directionality (duration-weighted) -> holder-weighted bond measure ---

# 1. One row per fund-bond-day with signed net (long +, short -)
fb = df_long_fund.merge(df_short_fund, on=['business_date','isin','fund_id'], how='outer')
fb[['long_vol','short_vol']] = fb[['long_vol','short_vol']].fillna(0)
fb['net']    = fb['long_vol'] - fb['short_vol']
fb['absnet'] = fb['net'].abs()

# 2. fund_dir per fund-day: DV01 (duration-weighted) directionality across ALL bonds
#    |sum net*dur| / sum|net*dur|.  0 = duration-hedged (carry or curve), 1 = directional.
fb = fb.merge(df_duration[['refdate','bondcode','duration']],
              left_on=['business_date','isin'], right_on=['refdate','bondcode'], how='left')
fb = fb.drop(columns=['refdate','bondcode'])
fb['dv01']    = fb['net'] * fb['duration']
fb['absdv01'] = fb['dv01'].abs()
fd = fb.groupby(['business_date','fund_id']).agg(
    net_dv01  =('dv01','sum'),
    gross_dv01=('absdv01','sum'),
).reset_index()
fd['fund_dir'] = fd['net_dv01'].abs() / fd['gross_dv01'].where(fd['gross_dv01'] > 0)
fund_dir = fd[['business_date','fund_id','fund_dir']]

# 3. holder_dir per bond-day: |net|-weighted average of holders' fund_dir.
#    Funds with undefined fund_dir (no duration) drop out of both num and den.
fb = fb.merge(fund_dir, on=['business_date','fund_id'], how='left')
fb['w_num']   = fb['absnet'] * fb['fund_dir']
fb['w_den']   = fb['absnet'].where(fb['fund_dir'].notna(), 0.0)
fb['islong']  = fb['long_vol']  > 0
fb['isshort'] = fb['short_vol'] > 0
holder = fb.groupby(['business_date','isin']).agg(
    holder_num =('w_num','sum'),
    holder_den =('w_den','sum'),
    gross_long =('long_vol','sum'),
    gross_short=('short_vol','sum'),
    n_long     =('islong','sum'),
    n_short    =('isshort','sum'),
).reset_index()
# safe divide: den<=0 becomes NaN so holder_dir is NaN there (stays float)
holder['holder_dir'] = holder['holder_num'] / holder['holder_den'].where(holder['holder_den'] > 0)
holder = holder.drop(columns=['holder_num','holder_den'])

# 4. merge onto out (holder_dir stays NaN where no HF holds the bond = zero-intensity baseline)
out = out.merge(holder, on=['business_date','isin'], how='left')


In [ ]:
out = out[['business_date', 'isin', 'PX_ASK', 'PX_BID', 'YLD_YTM_ASK',
       'YLD_YTM_BID', 'amt_issued', 'residual_bond_maturity',
       'collateral_country', 'borrowing_volume', 'long_maturity',
       'long_haircut', 'lending_volume', 'short_maturity', 'short_haircut',
       'net_pos', 'OIS_2Y', 'yld_mid', 'delta_y', 'abs_net',
       'hf_intensity_pre', 'net_pos_scaled', 'hf_intensity_long',
       'hf_intensity_short', 'bid_ask_spread', 'ctd_flag', 'hf_involved',
       'prev_net_pos', 'hf_involved_long', 'hf_involved_short', 'hf_category',
       'hf_category_ext', 'refdate', 'duration', 'date', 'pc1',
       'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median',
       'daily_net_change', 'delta_intensity', 'is_long_pre', 'is_short_pre',
       'placebo_shock', 'gross_long', 'n_long', 'gross_short', 'n_short', 'holder_dir']]

In [ ]:
out.to_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\monetary_policy_induced_position.csv')

# 2. Descriptive statistics → Table I

The `describe()` outputs below populate the Summary Statistics table (Panels A–D).

In [ ]:
out['delta_y'].describe()

In [ ]:
out['delta_y'][out['OIS_2Y'] == 0].describe()

In [ ]:
out['duration'].describe()

In [ ]:
out['bid_ask_spread'].describe()

In [ ]:
out['amt_issued'].describe()

In [ ]:
out['ctd_flag'].describe()

In [ ]:
out['hf_involved'].describe()

In [ ]:
out['hf_intensity_pre'].describe()

In [ ]:
out['hf_intensity_long'].describe()

In [ ]:
out['hf_intensity_short'].describe()

In [ ]:
round(out['hf_intensity_pre'][out['hf_intensity_pre'] != 0].describe(),2)

In [ ]:
round(out['hf_intensity_long'][out['hf_intensity_long'] != 0].describe(),2)

In [ ]:
round(out['hf_intensity_short'][out['hf_intensity_short'] != 0].describe(),2)

In [ ]:
round(out['long_haircut'][out['hf_intensity_long'] != 0].describe(),2)

In [ ]:
out['delta_y'][out['OIS_2Y'] == 0].abs().mean()

In [ ]:
round(out['short_haircut'][out['hf_intensity_short'] != 0].describe(),2)

In [ ]:
round(out['long_maturity'][out['hf_intensity_long'] != 0].describe(),2)

In [ ]:
round(out['short_maturity'][out['hf_intensity_short'] != 0].describe(),2)

In [ ]:
round(out['OIS_2Y'][(out['OIS_2Y'] != 0)].describe(),2)

In [ ]:
round(out['OIS_2Y'][(out['OIS_2Y'] > 0)].describe(),2)

In [ ]:
round(out['OIS_2Y'][(out['OIS_2Y'] < 0)].describe(),2)

In [ ]:
out['OIS_2Y'][out['OIS_2Y'] < 0].describe()

In [ ]:
out['OIS_2Y'][out['OIS_2Y'] > 0].describe()

# 3. Figures

## Figure 2 — Yield sensitivity to MP shocks (`shock_reaction_scatterplot`)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Clean Data
plot_data = out.copy()



# 2. Residualize Y
plot_data['delta_y_resid'] = (
    plot_data['delta_y'] - 
    plot_data.groupby('isin')['delta_y'].transform('mean')
)
plot_data['delta_y_resid'] += plot_data['delta_y'].mean()

# ---------------------------------------------------------
# 3. WINSORIZE (The Fix for Asymmetry)
low_cut = plot_data['OIS_2Y'][plot_data['OIS_2Y'] != 0].quantile(0.02)
high_cut = plot_data['OIS_2Y'][plot_data['OIS_2Y'] != 0].quantile(0.98)
plot_data = plot_data[(plot_data['OIS_2Y'] >= low_cut) & (plot_data['OIS_2Y'] <= high_cut)]

# Replace the winsorize step with clipping
#plot_data['OIS_2Y'] = plot_data['OIS_2Y'].clip(-18, 18)

# 4. Create Bins (Uniform width)
plot_data['bin'] = pd.cut(plot_data['OIS_2Y'], bins=18)
# ---------------------------------------------------------

# 5. Aggregate
binned_means = (plot_data.groupby(['hf_involved', 'bin'], observed=True)
                .agg({'OIS_2Y': 'mean', 'delta_y_resid': 'mean', 'delta_y': 'count'})
                .rename(columns={'delta_y': 'obs_count'})
                .reset_index())

# Filter for robustness (minimum 10 obs per dot)
#binned_means = binned_means[binned_means['obs_count'] > 10]

# 6. ENFORCE COMMON SUPPORT
# Drop bins where we don't have BOTH groups to compare
valid_bins = binned_means.groupby('bin', observed=True)['hf_involved'].nunique()
bins_with_both = valid_bins[valid_bins == 2].index
binned_means = binned_means[binned_means['bin'].isin(bins_with_both)]

# 7. Plot
sns.set_theme(style="ticks", context="paper", font_scale=1.2)
plt.figure(figsize=(10, 6))

# A. Regression Lines (using the WINSORIZED raw data)
sns.regplot(
    data=plot_data[plot_data['hf_involved']==0], 
    x='OIS_2Y', y='delta_y_resid', 
    scatter=False, color='gray', line_kws={'linestyle': '--', 'alpha': 0.8}, label='_nolegend_'
)
sns.regplot(
    data=plot_data[plot_data['hf_involved']==1], 
    x='OIS_2Y', y='delta_y_resid', 
    scatter=False, color='firebrick', label='_nolegend_'
)

# B. Binned Dots
sns.scatterplot(
    data=binned_means,
    x='OIS_2Y',
    y='delta_y_resid',
    hue='hf_involved',
    palette=['gray', 'firebrick'],
    style='hf_involved',
    s=100, 
    edgecolor='k',
    zorder=10
)

# Formatting
plt.xlabel('Monetary Policy Shock (bps)')
plt.ylabel('Daily Yield Change (bps)')
plt.axvline(0, color='black', linewidth=0.8, linestyle=':')
plt.axhline(plot_data['delta_y'].mean(), color='black', linewidth=0.8, linestyle=':')

# Custom Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='No Hedge Fund involved', 
           markerfacecolor='gray', markersize=10, markeredgecolor='k'),
    Line2D([0], [0], marker='X', color='w', label='Hedge Fund Involved', 
           markerfacecolor='firebrick', markersize=10, markeredgecolor='k')
]
plt.legend(handles=legend_elements, loc='upper left', frameon=True)

plt.tight_layout()
plt.savefig(r"C:\Users\hermesf\Projects\JobMarket\Figures\shock_reaction_scatterplot.png", dpi=300, bbox_inches="tight")
plt.show()

## Figure 3 — Exceedance probability by day type (`exceedance_probability`)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Prepare Data
out['abs_delta_y'] = out['delta_y'].abs()
out['day_type'] = out['OIS_2Y'].apply(lambda x: 'Shock Day' if x != 0 else 'Regular Day')

# 2. Define Continuous Thresholds
thresholds = np.linspace(0, 30, 301)

# 3. Calculate Exceedance Probabilities
results = []
shock_values = out[out['day_type'] == 'Shock Day']['abs_delta_y'].values
regular_values = out[out['day_type'] == 'Regular Day']['abs_delta_y'].values

for t in thresholds:
    results.append({
        'Threshold': t, 
        'Shock Day': np.mean(shock_values > t), 
        'Regular Day': np.mean(regular_values > t)
    })

df_plot = pd.DataFrame(results)

# 4. Plot
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.figure(figsize=(10, 6))

# A. Base Risk (Regular Days)
# We plot the Gray line and fill everything below it as "Standard Market Risk"
plt.plot(df_plot['Threshold'], df_plot['Regular Day'], 
         color='gray', label='Regular Day', linewidth=2)
plt.fill_between(df_plot['Threshold'], 0, df_plot['Regular Day'], 
                 color='gray', alpha=0.3)

# B. Excess Risk (Shock Days)
# We plot the Red line, but we ONLY fill the area BETWEEN Gray and Red.
# This visually isolates the "Marginal Contribution" of the shock.
plt.plot(df_plot['Threshold'], df_plot['Shock Day'], 
         color='firebrick', label='Shock Day', linewidth=2)

plt.fill_between(df_plot['Threshold'], df_plot['Regular Day'], df_plot['Shock Day'],
                 where=(df_plot['Shock Day'] >= df_plot['Regular Day']),
                 color='firebrick', alpha=0.3, interpolate=True)

# 5. Formatting
plt.xlabel('Absolute Daily Yield Change (bps)')
plt.ylabel('Exceedance Probability')
plt.xlim(0, 30)
plt.ylim(0, 1)
plt.legend(title='Day Type')

plt.tight_layout()
plt.savefig(r"C:\Users\hermesf\Projects\JobMarket\Figures\exceedance_probability.png", dpi=300, bbox_inches="tight")
plt.show()

## Figure 1 — Hedge fund net positions over time (`motivation_chart`)

In [ ]:
# Data prep
query = f""" 

SELECT s.business_date, 
security_isin as isin,
sum(nominal_euro) as borrowing_volume

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND (
(security_type = 'GOVS' AND LEFT(security_isin, 2) IN ('DE', 'IT', 'FR', 'ES') AND gnlcoll = 'SPEC' AND assttp_scty_issr_sector_riad = 'S1311') 
OR (LEFT(security_isin, 2) IN ('DE', 'IT', 'FR', 'ES') AND gnlcoll = 'SPEC' AND security_isin IN {unique_govs})) --EA GOVS) 
GROUP BY business_date, isin
ORDER BY business_date, isin 

""" 

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f""" 
 

SELECT s.business_date,
security_isin as isin,
sum(nominal_euro) as lending_volume

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'  
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND (
(security_type = 'GOVS' AND LEFT(security_isin, 2) IN ('DE', 'IT', 'FR', 'ES') AND gnlcoll = 'SPEC' AND assttp_scty_issr_sector_riad = 'S1311') 
OR (LEFT(security_isin, 2) IN ('DE', 'IT', 'FR', 'ES') AND gnlcoll = 'SPEC' AND security_isin IN {unique_govs})) --EA GOVS) 
GROUP BY business_date, isin
ORDER BY business_date, isin 

""" 

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df_repo = df_borrowing.merge( df_lending, on=['business_date', 'isin'], how='outer', suffixes=('_borrowing', '_lending') )
 

In [ ]:
df_repo = df_borrowing.merge( df_lending, on=['business_date', 'isin'], how='outer', suffixes=('_borrowing', '_lending') )
 

In [ ]:
df_repo['borrowing_volume'].fillna(0, inplace = True)
df_repo['lending_volume'].fillna(0, inplace = True)

In [ ]:
df_repo['net_pos'] = (df_repo['borrowing_volume'] - df_repo['lending_volume'])/1e9

In [ ]:
df_repo['net_pos'].fillna(0, inplace=True)

In [ ]:
df_repo['business_date'] = pd.to_datetime(df_repo['business_date'])

In [ ]:
df_repo['collateral_country'] = df_repo['isin'].str[:2]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. PREPARE THE DATA
df_repo['business_date'] = pd.to_datetime(df_repo['business_date'])

# Expand to all four countries
target_countries = ['DE', 'IT', 'ES', 'FR']
df_filtered = df_repo[df_repo['collateral_country'].isin(target_countries)].copy()

# Pivot the data
df_pivot = df_filtered.pivot_table(
    index='business_date', 
    columns='collateral_country', 
    values='net_pos', 
    aggfunc='sum'
).fillna(0)

# Reorder columns from core to periphery for consistent visual layering
col_order = [c for c in ['DE', 'FR', 'ES', 'IT'] if c in df_pivot.columns]
df_pivot = df_pivot[col_order]

# Separate positive and negative flows
df_pos = df_pivot.clip(lower=0)
df_neg = df_pivot.clip(upper=0)

# 2. AESTHETICS
colors_map = {
    'DE': '#5D6D7E',  # Slate grey (core)
    'FR': '#8B9DA8',  # Lighter grey-blue (semi-core)
    'ES': '#A89B6E',  # Muted gold (periphery)
    'IT': '#6E8E59'   # Muted olive green (periphery)
}
cols_in_plot = df_pivot.columns
plot_colors = [colors_map.get(c, '#888888') for c in cols_in_plot]

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

# 3. PLOT
fig, ax = plt.subplots(figsize=(10, 6))

ax.stackplot(df_pos.index, df_pos.T, labels=cols_in_plot, colors=plot_colors, alpha=0.85)
ax.stackplot(df_neg.index, df_neg.T, colors=plot_colors, alpha=0.85)

# 4. FORMATTING
ax.set_ylabel('Net Position (bln Eur)', fontsize=10, color='#333333')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#888888')
ax.spines['bottom'].set_color('#888888')

ax.grid(axis='y', color='#DDDDDD', linestyle='--', linewidth=0.7, zorder=0)
ax.axhline(0, color='black', linewidth=0.8, zorder=1)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
plt.xticks(rotation=0, fontsize=9, color='#333333')
plt.yticks(fontsize=9, color='#333333')

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), 
          frameon=False, loc='upper left', ncol=4, fontsize=10)

plt.tight_layout()
plt.savefig(r"C:\Users\hermesf\Projects\JobMarket\Figures\motivation_chart.png", dpi=300, bbox_inches="tight")
plt.show()

# 4. External validity

Share of hedge-fund activity by country (supports the footnote that DE + IT ≈ 70% of euro-area HF sovereign activity).

In [ ]:
# Data prep
query = f""" 

SELECT business_date, sum(nominal_euro) as borrowing_volume, LEFT(security_isin, 2) as cntr

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND (
(security_type = 'GOVS' AND LEFT(security_isin, 2) IN ('AT','BE','CY','DE','EE','ES','FI','FR','GR','IE','IT', 'LT','LU','LV','MT','NL','PT','SI','SK','HR') AND gnlcoll = 'SPEC' AND assttp_scty_issr_sector_riad = 'S1311') 
OR (LEFT(security_isin, 2) IN ('AT','BE','CY','DE','EE','ES','FI','FR','GR','IE','IT', 'LT','LU','LV','MT','NL','PT','SI','SK','HR') AND gnlcoll = 'SPEC' AND security_isin IN {unique_govs})) --EA GOVS) 
--AND contractual_maturity <= 1
GROUP BY business_date, cntr

""" 

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f""" 
 

SELECT 
business_date, sum(nominal_euro) as lending_volume, LEFT(security_isin, 2) as cntr

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'  
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND (
(security_type = 'GOVS' AND LEFT(security_isin, 2) IN ('AT','BE','CY','DE','EE','ES','FI','FR','GR','IE','IT', 'LT','LU','LV','MT','NL','PT','SI','SK','HR') AND gnlcoll = 'SPEC' AND assttp_scty_issr_sector_riad = 'S1311') 
OR (LEFT(security_isin, 2) IN ('AT','BE','CY','DE','EE','ES','FI','FR','GR','IE','IT', 'LT','LU','LV','MT','NL','PT','SI','SK','HR') AND gnlcoll = 'SPEC' AND security_isin IN {unique_govs})) --EA GOVS) 
--AND contractual_maturity <= 1
GROUP BY business_date, cntr

""" 

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df_repo = df_borrowing.merge( df_lending, on=['business_date','cntr'], how='outer', suffixes=('_borrowing', '_lending') )
 

In [ ]:
df_repo['borrowing_volume'].fillna(0, inplace = True)
df_repo['lending_volume'].fillna(0, inplace = True)

In [ ]:
df_repo['gross_position'] = (df_repo['borrowing_volume'] - df_repo['lending_volume']).abs()

In [ ]:
df_repo.head()

In [ ]:
df_repo = df_repo[['cntr', 'gross_position']].groupby('cntr').mean().reset_index()

In [ ]:
df_repo

In [ ]:
df_repo['share'] = (df_repo['gross_position']/df_repo['gross_position'].sum()) * 100

In [ ]:
df_repo

In [ ]:
df_repo.sort_values(by='share', ascending=False)